# KIS — SigLIP-2 keyframe embedding (Kaggle GPU)

Notebook này **chỉ làm bước embed**: đọc keyframe JPEG + `meta.db` từ Kaggle Dataset,
chạy SigLIP-2 trên GPU, ghi shard đúng format mà `kis index build` đọc được
(`embeddings/<pack>/<video_id>.{npy,json}`). Không cần upload video, không cần cài repo.

## Chuẩn bị (làm ở máy local)

```bash
# 1. Fold WAL vào meta.db trước khi copy
sqlite3 index/meta.db "PRAGMA wal_checkpoint(TRUNCATE)"

# 2. Đóng gói keyframe (chỉ keyframes, bỏ thumbnails)
tar -cf batch2_keyframes.tar -C index keyframes

# 3. Tạo dataset upload
mkdir -p kaggle_upload
mv batch2_keyframes.tar kaggle_upload/
cp index/meta.db kaggle_upload/
kaggle datasets init -p kaggle_upload   # sửa title/slug trong dataset-metadata.json
kaggle datasets create -p kaggle_upload
# lần sau: kaggle datasets version -p kaggle_upload -m "batch2 keyframes"
```

Trên Kaggle: attach dataset vừa tạo, bật **Internet** (tải model từ HF),
chọn accelerator **GPU T4 x2** (notebook chỉ dùng 1 GPU).

Chạy tuần tự hết notebook, cuối cùng tải `embeddings.zip` ở panel Output.

## 1. Cài dependency

Dùng torch CUDA có sẵn của Kaggle, chỉ nâng `transformers` (SigLIP-2 cần >= 4.50).
Không cài `torch` để tránh kéo wheel CPU/CUDA sai.

In [ ]:
%pip install -q "transformers>=4.50" sentencepiece

## 2. Cấu hình

`MODEL_ID` phải trùng `KIS_EMBED_MODEL` lúc build index ở local.
Đổi model thì phải embed lại **toàn bộ**, không trộn shard giữa 2 model.

In [ ]:
import glob
import hashlib
import json
import os
import shutil
import sqlite3
import subprocess
import sys
import tarfile
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

from tqdm.auto import tqdm

MODEL_ID = "google/siglip2-so400m-patch14-384"
# google/siglip2-base-patch16-384   -> 768-d,  batch 128
# google/siglip2-large-patch16-384  -> 1024-d, batch 64
BATCH = 32               # so400m trên T4 16GB; OOM thì tự giảm batch
OUTPUT_DTYPE = "float16"
DEVICE = "cuda"

WORK = Path("/kaggle/working")
INDEX = WORK / "index"
KEYFRAMES = INDEX / "keyframes"
EMBEDDINGS = INDEX / "embeddings"
DB_PATH = INDEX / "meta.db"

print("model:", MODEL_ID)
print("batch:", BATCH, "| dtype:", OUTPUT_DTYPE)
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip() or "NO GPU DETECTED")

## 3. Chuẩn bị thư mục index

Notebook chấp nhận dataset chứa folder `keyframes/` (symlink, không copy 6 GB)
hoặc file `keyframes*.tar*` (giải nén). `meta.db` được copy bằng API backup của
SQLite để đọc đúng cả WAL.

In [ ]:
def first_match(*patterns):
    for pattern in patterns:
        hits = sorted(glob.glob(pattern, recursive=True))
        if hits:
            return Path(hits[0])
    return None


def lookup_sample(db_path):
    """image_path đầu tiên trong DB; đọc qua bản copy vì mount Kaggle là read-only."""
    probe = INDEX / ".probe.db"
    shutil.copy2(db_path, probe)
    try:
        con = sqlite3.connect(probe)
        try:
            row = con.execute("SELECT image_path FROM keyframes LIMIT 1").fetchone()
        finally:
            con.close()
    except sqlite3.Error:
        row = None
    finally:
        probe.unlink(missing_ok=True)
    return Path(row[0]) if row else None


def resolve_dataset(roots, check):
    """Tìm (meta.db, thư mục keyframes) khớp nhau theo image_path.

    Kaggle tự giải nén .tar nên keyframes không nằm ngay dưới slug dataset
    (ví dụ .../kis-batch2-keyframes/batch2_keyframes/keyframes/...).
    """
    for base in roots:
        for db in sorted(glob.glob(f"{base}/**/meta.db", recursive=True)):
            rel = lookup_sample(Path(db))
            if rel is None:
                continue
            hits = sorted(glob.glob(f"{check}/**/{rel}", recursive=True))
            if hits:
                return Path(db), Path(hits[0]).parents[len(rel.parts) - 2]
    return None, None


INDEX.mkdir(parents=True, exist_ok=True)
SOURCE = Path("/kaggle/input")

db_src, kf_dir = resolve_dataset([SOURCE], SOURCE)
if db_src is None:
    tar_path = first_match(
        f"{SOURCE}/*/*keyframes*.tar*", f"{SOURCE}/*/*.tar.gz", f"{SOURCE}/*/*.tar",
    )
    if tar_path:
        print("extracting", tar_path, "->", INDEX)
        with tarfile.open(tar_path) as handle:
            if sys.version_info >= (3, 12):
                handle.extractall(INDEX, filter="data")
            else:
                handle.extractall(INDEX)
        db_src, kf_dir = resolve_dataset([SOURCE, INDEX], INDEX)
if db_src is None:
    raise SystemExit(f"không thấy cặp meta.db + keyframes khớp nhau trong {SOURCE}")

if not KEYFRAMES.exists():
    KEYFRAMES.symlink_to(kf_dir, target_is_directory=True)
    print("symlink:", KEYFRAMES, "->", kf_dir)

if not DB_PATH.exists():
    shutil.copy2(db_src, DB_PATH)
    wal_src = Path(str(db_src) + "-wal")
    if wal_src.exists():
        shutil.copy2(wal_src, Path(str(DB_PATH) + "-wal"))
    checkpoint = sqlite3.connect(DB_PATH)
    checkpoint.execute("PRAGMA wal_checkpoint(TRUNCATE)")
    checkpoint.close()
    print("copied:", db_src, "->", DB_PATH)

print("index dir:", INDEX)
print("keyframes:", KEYFRAMES, "| resolved:", Path(KEYFRAMES).resolve())


## 4. Preflight

Kiểm tra số video/keyframe và vài `image_path` có thật trên đĩa.

In [ ]:
conn = sqlite3.connect(DB_PATH)
n_videos = conn.execute("SELECT COUNT(*) FROM videos").fetchone()[0]
n_kf = conn.execute("SELECT COUNT(*) FROM keyframes").fetchone()[0]
n_kf_videos = conn.execute("SELECT COUNT(DISTINCT video_id) FROM keyframes").fetchone()[0]

print(f"videos: {n_videos} | videos có keyframe: {n_kf_videos} | keyframes: {n_kf}")

sample_paths = [row[0] for row in conn.execute(
    "SELECT image_path FROM keyframes ORDER BY keyframe_pk LIMIT 200"
)]
missing = [p for p in sample_paths if not (INDEX / p).exists()]
print("missing trong 200 mẫu:", len(missing))
if missing:
    print("ví dụ:", missing[:3])

## 5. Load model

Giữ weight fp32 + autocast fp16 (ổn định hơn ép cả model sang fp16).
so400m-patch14-384 có ~1,14B tham số (weight fp32 ~4,5 GB) nên batch mặc định 32;
cell embed tự giảm batch nếu OOM.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as functional
import transformers
from PIL import Image
from transformers import AutoModel, AutoProcessor

print("transformers:", transformers.__version__, "| torch:", torch.__version__)
assert torch.cuda.is_available(), "Chưa bật GPU: Settings -> Accelerator -> GPU T4 x2"

device = DEVICE
torch.backends.cudnn.benchmark = True

model = AutoModel.from_pretrained(MODEL_ID, attn_implementation="sdpa").eval().to(device)
processor = AutoProcessor.from_pretrained(MODEL_ID)

EMBED_DIM = int(model.config.text_config.projection_size)
print("embedding dim:", EMBED_DIM)
print("fp16 autocast:", OUTPUT_DTYPE == "float16")


def embed_images(paths):
    images = []
    for path in paths:
        with Image.open(path) as image:
            images.append(image.convert("RGB"))
    try:
        inputs = processor(images=images, return_tensors="pt").to(device)
        with torch.inference_mode(), torch.autocast(
            device_type="cuda", dtype=torch.float16,
            enabled=OUTPUT_DTYPE == "float16",
        ):
            output = model.get_image_features(**inputs)
        # transformers >= 5 wraps pooled features in BaseModelOutputWithPooling
        output = output.pooler_output if hasattr(output, "pooler_output") else output
        features = functional.normalize(output.float(), p=2, dim=-1)
        return features.cpu().numpy().astype(np.float16)
    finally:
        for image in images:
            image.close()

## 6. Smoke test

4 ảnh đầu tiên: kiểm tra shape, dim, norm ≈ 1.

In [ ]:
# smoke_paths = [INDEX / row[0] for row in conn.execute(
#     "SELECT image_path FROM keyframes ORDER BY keyframe_pk LIMIT 4"
# )]
# smoke = embed_images(smoke_paths)
# norms = np.linalg.norm(smoke.astype(np.float32), axis=1)
# print("shape:", smoke.shape, "| dtype:", smoke.dtype)
# print("norm:", np.round(norms, 5))
# assert smoke.shape[1] == EMBED_DIM
# assert np.all(np.abs(norms - 1.0) < 0.02)
# torch.cuda.empty_cache()

## 7. Embed toàn bộ (resume được)

Mỗi video ghi shard ngay khi xong; chạy lại cell này sẽ bỏ qua shard hợp lệ.
Nếu OOM, tự giảm batch cho video đó rồi thử lại.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def shard_paths(video_id, pack_id):
    pack = pack_id or video_id.split("_", 1)[0]
    base = EMBEDDINGS / pack
    return base / f"{video_id}.npy", base / f"{video_id}.json"


def shard_is_valid(npy_path, meta_path, keyframe_pks):
    if not npy_path.exists() or not meta_path.exists():
        return False
    try:
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return False
    return (
        meta.get("model") == MODEL_ID
        and meta.get("keyframe_pks") == keyframe_pks
        and int(meta.get("count", -1)) == len(keyframe_pks)
        and meta.get("sha256") == sha256_file(npy_path)
    )


def write_shard(npy_path, meta_path, array, video_id, keyframe_pks):
    npy_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_npy = npy_path.with_suffix(".tmp.npy")
    np.save(tmp_npy, array)
    os.replace(tmp_npy, npy_path)

    meta = {
        "video_id": video_id,
        "model": MODEL_ID,
        "dim": int(array.shape[1]),
        "dtype": OUTPUT_DTYPE,
        "count": int(array.shape[0]),
        "keyframe_pks": keyframe_pks,
        "sha256": sha256_file(npy_path),
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    tmp_meta = meta_path.with_name(meta_path.name + ".tmp")
    tmp_meta.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(tmp_meta, meta_path)


videos = conn.execute("SELECT video_id, pack_id FROM videos ORDER BY video_id").fetchall()
todo = []
total_images = 0
skipped = 0
for video_id, pack_id in videos:
    rows = conn.execute(
        "SELECT keyframe_pk, image_path FROM keyframes WHERE video_id = ? ORDER BY kf_n",
        (video_id,),
    ).fetchall()
    if not rows:
        continue
    keyframe_pks = [int(row[0]) for row in rows]
    npy_path, meta_path = shard_paths(video_id, pack_id)
    if shard_is_valid(npy_path, meta_path, keyframe_pks):
        skipped += 1
        continue
    todo.append((video_id, rows, keyframe_pks, npy_path, meta_path))
    total_images += len(rows)

print(f"cần embed: {len(todo)} video / {total_images} ảnh | đã có: {skipped} shard")

started = time.time()
done_images = 0
failures = []

for video_id, rows, keyframe_pks, npy_path, meta_path in tqdm(todo, desc="videos"):
    batch = BATCH
    try:
        while True:
            try:
                chunks = []
                for start in range(0, len(rows), batch):
                    paths = [INDEX / row[1] for row in rows[start : start + batch]]
                    chunks.append(embed_images(paths))
                array = np.concatenate(chunks, axis=0)
                break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                batch //= 2
                if batch < 4:
                    raise
                print(f"{video_id}: OOM, giảm batch còn {batch}")

        write_shard(npy_path, meta_path, array, video_id, keyframe_pks)
    except Exception as exc:
        failures.append((video_id, f"{type(exc).__name__}: {exc}"))
        tqdm.write(f"{video_id}: LỖI {type(exc).__name__}: {exc} (bỏ qua, chạy lại để thử)")
        continue

    done_images += int(array.shape[0])

    elapsed = time.time() - started
    rate = done_images / elapsed if elapsed > 0 else 0.0
    tqdm.write(
        f"{video_id}: {array.shape[0]} vec | {rate:.1f} img/s "
        f"| còn {total_images - done_images} ảnh"
    )

seconds = time.time() - started
print(
    f"xong: {done_images} ảnh trong {seconds / 60:.1f} phút "
    f"({done_images / seconds if seconds else 0:.1f} img/s), lỗi: {len(failures)}"
)

## 8. Kiểm tra shard trước khi tải về

Đối chiếu với `meta.db`: đủ `keyframe_pk`, đúng dim, checksum khớp, vector đã normalize.

In [ ]:
db_pks = {int(row[0]) for row in conn.execute("SELECT keyframe_pk FROM keyframes")}
shard_pks = set()
total_vectors = 0
dims = set()
problems = []

meta_files = sorted(EMBEDDINGS.glob("*/*.json"))
for meta_path in meta_files:
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    npy_path = meta_path.with_suffix(".npy")
    if not npy_path.exists():
        problems.append(f"thiếu {npy_path}")
        continue
    if meta.get("model") != MODEL_ID:
        problems.append(f"{meta_path}: model {meta.get('model')}")
    if meta.get("sha256") != sha256_file(npy_path):
        problems.append(f"{meta_path}: checksum lệch")
    array = np.load(npy_path, mmap_mode="r")
    dims.add(int(array.shape[1]))
    total_vectors += int(array.shape[0])
    shard_pks.update(int(p) for p in meta.get("keyframe_pks", []))

missing = db_pks - shard_pks
extra = shard_pks - db_pks
print(f"shards: {len(meta_files)} | vectors: {total_vectors} | dims: {dims}")
print(f"keyframe_pks: DB {len(db_pks)} | shards {len(shard_pks)} | thiếu {len(missing)} | thừa {len(extra)}")
print("problems:", len(problems))
for item in problems[:5]:
    print(" -", item)

if meta_files:
    probe = np.load(meta_files[0].with_suffix(".npy")).astype(np.float32)
    norms = np.linalg.norm(probe, axis=1)
    print("norm min/max:", float(norms.min()), float(norms.max()))

summary = {
    "model": MODEL_ID,
    "dim": sorted(dims)[0] if dims else None,
    "dtype": OUTPUT_DTYPE,
    "shards": len(meta_files),
    "vectors": total_vectors,
    "db_keyframes": len(db_pks),
    "missing": len(missing),
    "extra": len(extra),
    "created_at": datetime.now(timezone.utc).isoformat(),
}
(WORK / "embed_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(summary, ensure_ascii=False, indent=2))

## 9. Đóng gói để tải về

`embeddings.zip` chứa `embeddings/...` (giữ nguyên cấu trúc pack/video).

In [ ]:
zip_path = WORK / "embeddings.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as archive:
    for path in sorted(EMBEDDINGS.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(INDEX))

print("zip:", zip_path, f"{zip_path.stat().st_size / 1e6:.1f} MB")
print("shard:", len(list(EMBEDDINGS.glob("*/*.npy"))), "video")

## 10. Làm ở local sau khi tải zip

```bash
# Giải nén vào index/ (ghi đè shard cũ của cùng model)
unzip -o embeddings.zip -d index/

# Build FAISS strict (đủ 100% keyframe) + manifest
uv run kis index build

# Search
uv run kis search "a woman in a white shirt" --k 20
```

Lưu ý:

- `KIS_EMBED_MODEL` lúc build phải đúng `MODEL_ID` ở trên (mặc định khớp).
- Đừng chạy `kis keyframes --force` sau khi đã embed: `keyframe_pk` đổi thì shard
  không còn khớp DB, `index build` sẽ báo lỗi và phải embed lại.
- Nếu đổi model (base/large/giant), phải embed lại **tất cả** video bằng model đó
  rồi build lại index; dim trong manifest sẽ khác và shard cũ không dùng chung được.